### Codigo 02 - Comparando modelo normal vs class_weight='balanced'

Neste notebook, vamos estudar uma estrategia simples para lidar com dados desbalanceados: usar pesos nas classes. A ideia e comparar um RandomForestClassifier normal com outro usando class_weight='balanced'.

Exemplo bancario: em uma base de fraude, a classe fraude costuma ser muito menor do que a classe transacao normal. Com class_weight, o modelo passa a dar mais importancia para erros cometidos nas classes menores.

- 1. Importaçoes

Primeiro importamos as bibliotecas principais. Vamos usar pandas para manipular a base, scikit-learn para treinar o modelo e algumas metricas para avaliar o desempenho.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split

### 2. Configuracoes da base

O curso espera um arquivo chamado cov_types.csv. Se ele nao existir na pasta do notebook, vamos tentar baixar a base Covertype oficial da UCI e salvar com esse nome.

Tambem definimos os nomes das colunas, porque a base original da UCI vem sem cabecalho.

In [2]:
CSV_PATH = Path("cov_types.csv")
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz"
RANDOM_SEED = 42
SAMPLE_SIZE = 50_000

COLUNAS = ([
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]
+ [f"Wilderness_Area_{i}" for i in range(1,5)]
+ [f"Soil_Type_{i}" for i in range(1,41)]
+ ["Cover_Type"]
)


### 3. Funcao para carregar a base

Esta funcao deixa o notebook mais facil de reutilizar. Ela procura o arquivo cov_types.csv. Se encontrar, carrega o arquivo local. Se nao encontrar, tenta baixar a base original e salvar localmente.
Isso resolve o problema comum de cursos em que o codigo existe, mas o arquivo de dados nao foi disponibilizado.

In [3]:
def carregar_base() -> pd.DataFrame:
    """Carrega a base local ou baixa a base oficial do UCI."""
    if not CSV_PATH.exists():
        print(f"Lendo arquivo local: {CSV_PATH.resolve()}")
        return pd.read_csv(CSV_PATH)
    
    print(f"Arquivo cov_types.csv não encontrado. ")
    print(f"Baixando base do UCI: {UCI_URL}")
    df = pd.read_csv(UCI_URL, header=None, names=COLUNAS, compression="gzip")
    df.to_csv(CSV_PATH, index=False)

    print(f"Base baixada e salva localmente em: {CSV_PATH.resolve()}")
    return df

### 4. Carregando e inspecionando os dados

Agora carregamos a base, observamos o tamanho e visualizamos as primeiras linhas. Se a base estiver muito grande, usamos uma amostra para o estudo ficar mais rapido.
Em um projeto real, voce pode remover a amostragem e treinar com a base completa.

In [4]:
df = carregar_base()

print("Formato original:", df.shape)
display(df.head())

if len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_SEED)
    print(f"Base amostrada para {SAMPLE_SIZE} registros.")

print("Formato após amostragem:", df.shape)

Arquivo cov_types.csv não encontrado. 
Baixando base do UCI: https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz
Base baixada e salva localmente em: C:\Users\Olive\VS Code\Python\meu-projeto\machine_learning_and_Data_Science\9_dados_desbalanceados\cov_types.csv
Formato original: (581012, 55)


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area_1,Wilderness_Area_2,Wilderness_Area_3,Wilderness_Area_4,Soil_Type_1,Soil_Type_2,Soil_Type_3,Soil_Type_4,Soil_Type_5,Soil_Type_6,Soil_Type_7,Soil_Type_8,Soil_Type_9,Soil_Type_10,Soil_Type_11,Soil_Type_12,Soil_Type_13,Soil_Type_14,Soil_Type_15,Soil_Type_16,Soil_Type_17,Soil_Type_18,Soil_Type_19,Soil_Type_20,Soil_Type_21,Soil_Type_22,Soil_Type_23,Soil_Type_24,Soil_Type_25,Soil_Type_26,Soil_Type_27,Soil_Type_28,Soil_Type_29,Soil_Type_30,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39,Soil_Type_40,Cover_Type
0,2596,51,3,258,0,510,221,232,148,6279,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
3,2785,155,18,242,118,3090,238,238,122,6211,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2
4,2595,45,2,153,-1,391,220,234,150,6172,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5


Base amostrada para 50000 registros.
Formato após amostragem: (50000, 55)
